<a href="https://colab.research.google.com/github/Alt4660/con5200-card-replication/blob/main/notebooks/02_robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

## Setup

In [1]:
import sys, os, subprocess
if "google.colab" in sys.modules and not os.path.exists("download_data.py"):
    subprocess.run(["git", "clone", "https://github.com/Alt4660/con5200-card-replication"], check=True)
    os.chdir("con5200-card-replication")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "linearmodels"], check=True)

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [4]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    raw = pd.read_stata(paths[0])
    print(raw.shape)
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

downloading https://raw.githubusercontent.com/scunning1975/mixtape/master/card.dta
  data/card.dta  0.5 MB
Card (1995), proximity to college and returns to schooling
(3010, 37)


In [ ]:
from linearmodels.iv import IV2SLS

REGIONS = " + ".join(f"reg66{i}" for i in range(2, 10))

def build(raw, cuts=None, labels=None):
    df = raw.copy()
    df["agesq"] = df["age"] ** 2
    for col in ["fatheduc", "motheduc"]:
        df[f"{col}_miss"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(df[col].mean())
    family = "fatheduc + motheduc + fatheduc_miss + motheduc_miss + momdad14 + sinmom14"
    if cuts is not None:
        mom = pd.cut(df["motheduc"], cuts, labels=labels).astype(str)
        dad = pd.cut(df["fatheduc"], cuts, labels=labels).astype(str)
        dummies = pd.get_dummies(mom + "_" + dad, prefix="pe", drop_first=True).astype(int)
        df = pd.concat([df, dummies], axis=1)
        family += " + " + " + ".join(dummies.columns)
    return df, f"black + south + smsa + smsa66 + {REGIONS} + {family}"


df, CONTROLS = build(raw, [-1, 11, 12, 30], ["lt12", "hs", "morehs"])

SIMPLE = f"lwage ~ 1 + exper + expersq + {CONTROLS} + [educ ~ nearc4]"

## The four exercises

### Exercise 1: Inference that does not need a strong first stage

**What I did.** *TODO*

In [ ]:
# Exercise 1: does the answer survive inference that does not rely on a strong first stage?
rows = []
for label, cov in [("2SLS, default standard errors", "unadjusted"), ("2SLS, robust standard errors", "robust")]:
    r = IV2SLS.from_formula(SIMPLE, data=df).fit(cov_type=cov)
    lo, hi = r.conf_int().loc["educ"]
    rows.append([label, r.params["educ"], r.std_errors["educ"], lo, hi])


def ar_pvalue(b0):
    d = df.assign(y0=df["lwage"] - b0 * df["educ"])
    m = smf.ols(f"y0 ~ nearc4 + exper + expersq + {CONTROLS}", data=d).fit(cov_type="HC1")
    return m.pvalues["nearc4"]

grid = np.round(np.arange(-0.10, 0.60, 0.002), 3)
kept = [b for b in grid if ar_pvalue(b) > 0.05]
rows.append(["Anderson-Rubin 95% set", r.params["educ"], np.nan, min(kept), max(kept)])

ex1 = pd.DataFrame(rows, columns=["method", "estimate", "se", "ci_low", "ci_high"])
ex1.to_csv(ROOT / "output" / "ex1_weak_iv_inference.csv", index=False)
print("number of grid points kept:", len(kept), "of a possible", int(round((max(kept) - min(kept)) / 0.002)) + 1)
ex1.round(3)

What I found: Using nearc4 as the only instrument, the IV return is 0.140. The usual 95% confidence interval is about 0.036 to 0.244, and the robust interval is almost the same. The Anderson-Rubin interval is slightly wider, going from about 0.044 to 0.280, but it still stays above zero.

What it implies about the original claim: The first-stage F-statistic is 16.19, so the instrument seems strong enough that weak-instrument concerns do not change the main result. Card's finding of a positive return to education is supported. However, this does not prove that the IV estimate is higher than OLS. My OLS estimate of 0.072 falls inside all three intervals, so I cannot say that the true return is higher than OLS. This also fits with Card's point that his IV estimates are not very precise.

### Exercise 2: Specification curve over the choices the paper leaves open

**What I did.** *TODO*

In [ ]:
# Exercise 2: The paper does not explain how it grouped parent education, whether experience is endogenous, or which standard errors it used.
# I test all the different combinations to see how much the IV return changes.
schemes = {
    "no interaction dummies": (None, None),
    "<12 / 12 / >12": ([-1, 11, 12, 30], ["a", "b", "c"]),
    "<12 / 12-15 / 16+": ([-1, 11, 15, 30], ["a", "b", "c"]),
    "<12 / 12 / 13-15 / 16+": ([-1, 11, 12, 15, 30], ["a", "b", "c", "d"]),
    "<10 / 10-11 / 12 / >12": ([-1, 9, 11, 12, 30], ["a", "b", "c", "d"]),
}
rows = []
for name, (cuts, labs) in schemes.items():
    d, C = build(raw, cuts, labs)
    for exper_endog in [True, False]:
        if exper_endog:
            f = f"lwage ~ 1 + {C} + [educ + exper + expersq ~ nearc4 + age + agesq]"
        else:
            f = f"lwage ~ 1 + exper + expersq + {C} + [educ ~ nearc4]"
        for cov in ["unadjusted", "robust"]:
            r = IV2SLS.from_formula(f, data=d).fit(cov_type=cov)
            rows.append([name, "endogenous" if exper_endog else "exogenous", cov,
                         r.params["educ"], r.std_errors["educ"]])

curve = pd.DataFrame(rows, columns=["parent_classes", "experience", "std_errors", "iv_return", "se"])
curve.to_csv(ROOT / "output" / "ex2_spec_curve.csv", index=False)

order = curve.sort_values("iv_return").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(order.index, order["iv_return"], yerr=1.96 * order["se"], fmt="o", ms=4, lw=1, color="0.3")
ax.axhline(0.132, color="C3", ls="--", label="Paper's IV (0.132)")
ax.axhline(0.073, color="C0", ls=":", label="Paper's OLS (0.073)")
ax.set_xlabel("Specification (sorted by the IV return)")
ax.set_ylabel("IV return to a year of schooling")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ROOT / "output" / "ex2_spec_curve.png", dpi=200)
print("smallest and largest IV across", len(curve), "specifications:",
      round(curve["iv_return"].min(), 4), round(curve["iv_return"].max(), 4))
curve.round(4)

**What I found.** *TODO*

**What it implies about the original claim.** *TODO: this part is what is graded.*

### Exercise 3: Balance check on the instrument

**What I did.** *TODO*

**What I found.** *TODO*

**What it implies about the original claim.** *TODO: this part is what is graded.*

### Exercise 4: How big a direct effect would undo the IV result

**What I did.** *TODO*

**What I found.** *TODO*

**What it implies about the original claim.** *TODO: this part is what is graded.*